# The objective of this notebook is to test the modularity of the exactBO library

## 0 - Imports

In [1]:
import tamubo.exactbo as ebo
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel

## 1 - ExactBO Loop Class

### 1.1 - Inputs

In [2]:
# Define model
kernel = ConstantKernel(1.0, (1e-3, 1e10)) * RBF(length_scale=0.5, length_scale_bounds=(10, 1e4))
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, normalize_y=True)

# Define bounds
bounds = [[-20,20],[-20,20]]

# Define precision
precision = 0.01

### 1.2 - Create object

In [3]:
eboloop = ebo.ExactBOLoop(gp, bounds, precision, log=True)

### 1.3 - Create true function and set it in object

In [4]:
# Function to minimize
def f(X):
    x, y = X[:,0], X[:,1]

    # Parameters
    alpha = 0.02
    A  = np.array([4.0, 3.0, 2.0])
    B  = np.array([8.0, 5.0, 2.0])    # betas
    C  = np.array([[ 12.0, -3.0],      # centers (x1,y1)
                [-1.0,  15.0],
                [-6.0, -7.0]])
    
    # Compute function value
    val = alpha*(x**2 + y**2)
    for Ai, Bi, (xi, yi) in zip(A, B, C):
        r2 = (x - xi)**2 + (y - yi)**2
        val -= Ai * np.exp(-r2 / Bi)
    return val

# Set in object
eboloop.set_oracle(f)

### 1.4 - Create initial point and evaluate

In [5]:
X0 = np.array([[0,0],[-10,-10],[-10,10],[10,-10],[10,10],[-20,-20],[20,20],[-20,20],[20,-20]])
y0 = f(X0)

### 1.5 - Run optimization (commented to test partition loop)

In [6]:
res = eboloop.run(X0,y0,10)
res

c:\Users\juan.florez\tamubo\venvs\venvEBO\Lib\site-packages\sklearn\gaussian_process\_gpr.py:663: ConvergenceWarning: lbfgs failed to converge after 13 iteration(s) (status=2):
ABNORMAL: 

You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  _check_optimize_result("lbfgs", opt_res)
c:\Users\juan.florez\tamubo\venvs\venvEBO\Lib\site-packages\sklearn\gaussian_process\_gpr.py:663: ConvergenceWarning: lbfgs failed to converge after 15 iteration(s) (status=2):
ABNORMAL: 

You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  _check_optimize_result("lbfgs", opt_res)
c:\Users\juan.florez\tamubo\venvs\venvEBO\Lib\site-packages\sklearn\gaussian_process\_gpr.py:663: ConvergenceWarning: lbfgs failed to converge after 12 iteration(s) (status=2):
ABNORMAL: 

You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  _ch

BOResult(X=array([[ 0.00000000e+00,  0.00000000e+00],
       [-1.00000000e+01, -1.00000000e+01],
       [-1.00000000e+01,  1.00000000e+01],
       [ 1.00000000e+01, -1.00000000e+01],
       [ 1.00000000e+01,  1.00000000e+01],
       [-2.00000000e+01, -2.00000000e+01],
       [ 2.00000000e+01,  2.00000000e+01],
       [-2.00000000e+01,  2.00000000e+01],
       [ 2.00000000e+01, -2.00000000e+01],
       [ 1.56250000e-01, -8.90625000e+00],
       [-1.98437500e+01,  4.68750000e-01],
       [ 1.56250000e-01, -1.56250000e-01],
       [ 1.56250000e-01, -1.56250000e-01],
       [-7.10542736e-14, -7.10542736e-14],
       [ 1.56250000e-01, -1.56250000e-01],
       [ 1.56250000e-01, -1.56250000e-01],
       [ 1.56250000e-01, -1.56250000e-01],
       [ 1.56250000e-01, -1.56250000e-01],
       [-1.56250000e-01,  1.56250000e-01]]), y=array([-1.97778020e-08,  3.99999255e+00,  4.00000000e+00,  3.99469288e+00,
        4.00000000e+00,  1.60000000e+01,  1.60000000e+01,  1.60000000e+01,
        1.60000000

In [7]:
ebolog = eboloop.log
ebolog.keys()

dict_keys(['ebo_log', 'start', 'ebo_it0', 'ebo_it1', 'ebo_it2', 'ebo_it3', 'ebo_it4', 'ebo_it5', 'ebo_it6', 'ebo_it7', 'ebo_it8', 'ebo_it9', 'result'])

In [8]:
ebolog['start']['domain']

array([[-20.,  20.],
       [-20.,  20.]])

In [8]:
ebolog['ebo_it0'].keys()

dict_keys(['start', 'ploop_start', 'ploop_0', 'ploop_1', 'ploop_2', 'ploop_3', 'ploop_4', 'ploop_5', 'ploop_6', 'ploop_7', 'ploop_final'])

In [10]:
ebolog['ebo_it0']['start'].X

array([[  0,   0],
       [-10, -10],
       [-10,  10],
       [ 10, -10],
       [ 10,  10],
       [-20, -20],
       [ 20,  20],
       [-20,  20],
       [ 20, -20]])

In [10]:
ebolog['ebo_it0']['ploop_start']

{'boxes': [Box(bounds=array([[-20.,  20.],
         [-20.,  20.]]), sampled=True, active=True, ei=None)],
 'best_x': None,
 'max_ei': 0.0}

In [11]:
ebolog['ebo_it0']['ploop_1']

{'boxes': [Box(bounds=array([[-20.,   0.],
         [-20.,   0.]]), sampled=True, active=True, ei=None),
  Box(bounds=array([[-20.,   0.],
         [  0.,  20.]]), sampled=True, active=True, ei=None),
  Box(bounds=array([[  0.,  20.],
         [-20.,   0.]]), sampled=True, active=True, ei=None),
  Box(bounds=array([[ 0., 20.],
         [ 0., 20.]]), sampled=True, active=True, ei=None)],
 'best_x': array([ 0.8, -9.6]),
 'max_ei': np.float64(0.6086143893908547)}

In [12]:
ebolog['ebo_it0']['ploop_2']

{'boxes': [Box(bounds=array([[-20., -10.],
         [-20., -10.]]), sampled=True, active=True, ei=None),
  Box(bounds=array([[-20., -10.],
         [-10.,   0.]]), sampled=True, active=True, ei=None),
  Box(bounds=array([[-10.,   0.],
         [-20., -10.]]), sampled=True, active=True, ei=None),
  Box(bounds=array([[-10.,   0.],
         [-10.,   0.]]), sampled=True, active=True, ei=None),
  Box(bounds=array([[-20., -10.],
         [  0.,  10.]]), sampled=True, active=True, ei=None),
  Box(bounds=array([[-20., -10.],
         [ 10.,  20.]]), sampled=True, active=True, ei=None),
  Box(bounds=array([[-10.,   0.],
         [  0.,  10.]]), sampled=True, active=True, ei=None),
  Box(bounds=array([[-10.,   0.],
         [ 10.,  20.]]), sampled=True, active=True, ei=None),
  Box(bounds=array([[  0.,  10.],
         [-20., -10.]]), sampled=True, active=True, ei=None),
  Box(bounds=array([[  0.,  10.],
         [-10.,   0.]]), sampled=True, active=True, ei=None),
  Box(bounds=array([[ 10.,  20.

In [13]:
ebolog['ebo_it0']['ploop_3']

{'boxes': [Box(bounds=array([[-20., -15.],
         [-20., -15.]]), sampled=True, active=True, ei=None),
  Box(bounds=array([[-20., -15.],
         [-15., -10.]]), sampled=False, active=True, ei=Bounds(lo=np.float64(-2.6998115974287153), hi=np.float64(2.2704100691953655))),
  Box(bounds=array([[-15., -10.],
         [-20., -15.]]), sampled=False, active=True, ei=Bounds(lo=np.float64(-2.6896878546207343), hi=np.float64(2.2721647598511385))),
  Box(bounds=array([[-15., -10.],
         [-15., -10.]]), sampled=True, active=True, ei=None),
  Box(bounds=array([[-20., -15.],
         [-10.,  -5.]]), sampled=False, active=True, ei=Bounds(lo=np.float64(-1.0571517161395967), hi=np.float64(2.3794966616390045))),
  Box(bounds=array([[-20., -15.],
         [ -5.,   0.]]), sampled=False, active=True, ei=Bounds(lo=np.float64(-0.29588400647143975), hi=np.float64(2.9787954515022115))),
  Box(bounds=array([[-15., -10.],
         [-10.,  -5.]]), sampled=True, active=True, ei=None),
  Box(bounds=array([[-

In [14]:
ebolog['ebo_it0']['ploop_4']

{'boxes': [Box(bounds=array([[-20. , -17.5],
         [-20. , -17.5]]), sampled=True, active=True, ei=None),
  Box(bounds=array([[-20. , -17.5],
         [-17.5, -15. ]]), sampled=False, active=True, ei=Bounds(lo=np.float64(-4.885827270263277), hi=np.float64(1.6739035660168335))),
  Box(bounds=array([[-17.5, -15. ],
         [-20. , -17.5]]), sampled=False, active=True, ei=Bounds(lo=np.float64(-4.884387808947945), hi=np.float64(1.6743971576881538))),
  Box(bounds=array([[-17.5, -15. ],
         [-17.5, -15. ]]), sampled=False, active=True, ei=Bounds(lo=np.float64(-4.35075085511452), hi=np.float64(1.7335933053404555))),
  Box(bounds=array([[-20. , -17.5],
         [-15. , -12.5]]), sampled=False, active=True, ei=Bounds(lo=np.float64(-4.377390831671501), hi=np.float64(1.8607508885081716))),
  Box(bounds=array([[-20. , -17.5],
         [-12.5, -10. ]]), sampled=False, active=True, ei=Bounds(lo=np.float64(-3.7555773189457935), hi=np.float64(1.9927681026434538))),
  Box(bounds=array([[-17.5

In [15]:
ebolog['ebo_it0']['ploop_5']

{'boxes': [Box(bounds=array([[-20.  , -18.75],
         [-20.  , -18.75]]), sampled=True, active=True, ei=None),
  Box(bounds=array([[-20.  , -18.75],
         [-18.75, -17.5 ]]), sampled=False, active=True, ei=Bounds(lo=np.float64(-5.084690513459005), hi=np.float64(1.0440085655597966))),
  Box(bounds=array([[-18.75, -17.5 ],
         [-20.  , -18.75]]), sampled=False, active=True, ei=Bounds(lo=np.float64(-5.085140434754706), hi=np.float64(1.0442326171401168))),
  Box(bounds=array([[-18.75, -17.5 ],
         [-18.75, -17.5 ]]), sampled=False, active=True, ei=Bounds(lo=np.float64(-5.085667859445753), hi=np.float64(1.1272720495705362))),
  Box(bounds=array([[-20.  , -18.75],
         [-17.5 , -16.25]]), sampled=False, active=True, ei=Bounds(lo=np.float64(-5.168239375677849), hi=np.float64(1.197082889012911))),
  Box(bounds=array([[-20.  , -18.75],
         [-16.25, -15.  ]]), sampled=False, active=True, ei=Bounds(lo=np.float64(-5.147915694009406), hi=np.float64(1.3511250720646528))),
  B

In [16]:
ebolog['ebo_it0']['ploop_6']

{'boxes': [Box(bounds=array([[-20.   , -19.375],
         [-20.   , -19.375]]), sampled=True, active=True, ei=None),
  Box(bounds=array([[-20.   , -19.375],
         [-19.375, -18.75 ]]), sampled=False, active=False, ei=Bounds(lo=np.float64(-4.245269048626584), hi=np.float64(0.6108561748679061))),
  Box(bounds=array([[-19.375, -18.75 ],
         [-20.   , -19.375]]), sampled=False, active=False, ei=Bounds(lo=np.float64(-4.246062226453112), hi=np.float64(0.610976540576485))),
  Box(bounds=array([[-19.375, -18.75 ],
         [-19.375, -18.75 ]]), sampled=False, active=True, ei=Bounds(lo=np.float64(-4.403654431758954), hi=np.float64(0.6581725633157419))),
  Box(bounds=array([[-20.   , -19.375],
         [-18.75 , -18.125]]), sampled=False, active=True, ei=Bounds(lo=np.float64(-4.51032569341917), hi=np.float64(0.6914550619389925))),
  Box(bounds=array([[-20.   , -19.375],
         [-18.125, -17.5  ]]), sampled=False, active=True, ei=Bounds(lo=np.float64(-4.751796553521691), hi=np.float64(0

In [17]:
ebolog['ebo_it0']['ploop_7']

{'boxes': [Box(bounds=array([[-20.    , -19.6875],
         [-20.    , -19.6875]]), sampled=True, active=True, ei=None),
  Box(bounds=array([[-20.    , -19.6875],
         [-19.6875, -19.375 ]]), sampled=False, active=True, ei=None),
  Box(bounds=array([[-19.6875, -19.375 ],
         [-20.    , -19.6875]]), sampled=False, active=True, ei=None),
  Box(bounds=array([[-19.6875, -19.375 ],
         [-19.6875, -19.375 ]]), sampled=False, active=True, ei=None),
  Box(bounds=array([[-20.   , -19.375],
         [-19.375, -18.75 ]]), sampled=False, active=False, ei=Bounds(lo=np.float64(-4.245269048626584), hi=np.float64(0.6108561748679061))),
  Box(bounds=array([[-19.375, -18.75 ],
         [-20.   , -19.375]]), sampled=False, active=False, ei=Bounds(lo=np.float64(-4.246062226453112), hi=np.float64(0.610976540576485))),
  Box(bounds=array([[-19.375 , -19.0625],
         [-19.375 , -19.0625]]), sampled=False, active=True, ei=None),
  Box(bounds=array([[-19.375 , -19.0625],
         [-19.0625, -1

In [18]:
ebolog['ebo_it0']['ploop_final']

{'boxes': [Box(bounds=array([[-20.    , -19.6875],
         [-20.    , -19.6875]]), sampled=True, active=True, ei=None),
  Box(bounds=array([[-20.    , -19.6875],
         [-19.6875, -19.375 ]]), sampled=False, active=True, ei=None),
  Box(bounds=array([[-19.6875, -19.375 ],
         [-20.    , -19.6875]]), sampled=False, active=True, ei=None),
  Box(bounds=array([[-19.6875, -19.375 ],
         [-19.6875, -19.375 ]]), sampled=False, active=True, ei=None),
  Box(bounds=array([[-20.   , -19.375],
         [-19.375, -18.75 ]]), sampled=False, active=False, ei=Bounds(lo=np.float64(-4.245269048626584), hi=np.float64(0.6108561748679061))),
  Box(bounds=array([[-19.375, -18.75 ],
         [-20.   , -19.375]]), sampled=False, active=False, ei=Bounds(lo=np.float64(-4.246062226453112), hi=np.float64(0.610976540576485))),
  Box(bounds=array([[-19.375 , -19.0625],
         [-19.375 , -19.0625]]), sampled=False, active=True, ei=None),
  Box(bounds=array([[-19.375 , -19.0625],
         [-19.0625, -1

## 2 - Partition Loop Class

### 2.1 - Train model

In [7]:
eboloop.model.fit(X0, y0.ravel())

/Users/juanesfco/tamubo/venvs/venvEBO/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__length_scale is close to the specified lower bound 1.0. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


,kernel,1**2 * RBF(length_scale=0.5)
,alpha,1e-06
,optimizer,'fmin_l_bfgs_b'
,n_restarts_optimizer,0
,normalize_y,True
,copy_X_train,True
,n_targets,None
,random_state,None
,kernel__k1,1**2
,kernel__k2,RBF(length_scale=0.5)
,kernel__k1__constant_value,1.0


### 2.2 - Create partition loop class

In [8]:
ploop = ebo.PartitionMaxEISearch(eboloop.model,eboloop.init_box,eboloop.grid, eboloop.precision)

In [9]:
ploop.boxes, ploop.best_x, ploop.max_ei

([Box(bounds=array([[-20.,  20.],
         [-20.,  20.]]), sampled=True, active=True)],
 None,
 0.0)

### 2.3 - Run a couple iterations of partition loop until it cannot longer partition

In [10]:
ploop.run(1)
len(ploop.boxes), ploop.best_x, ploop.max_ei

(4, array([-20.,  20.]), np.float64(0.01358068304274647))

In [11]:
ploop.run(1)
len(ploop.boxes), ploop.best_x, ploop.max_ei

(16, array([ 5., -5.]), np.float64(0.013580683043505737))

In [12]:
ploop.run(1)
len(ploop.boxes), ploop.best_x, ploop.max_ei

(64, array([-2.5, -2.5]), np.float64(0.013721657794326256))

In [13]:
ploop.run(1)
len(ploop.boxes), ploop.best_x, ploop.max_ei

(256, array([-1.25, -1.25]), np.float64(0.03494242889846291))

In [14]:
ploop.run(1)
len(ploop.boxes), ploop.best_x, ploop.max_ei

(1024, array([-0.625, -0.625]), np.float64(0.12305104912315462))

In [15]:
ploop.run(1)
len(ploop.boxes), ploop.best_x, ploop.max_ei

(1024, array([-0.625, -0.625]), np.float64(0.12305104912315462))

## 3 - Testing Stuff

In [5]:
#hola = False
hola = {'si':'yes'}
#hola = {}
if hola:
    print("si")
else:
    print("no")

si


In [9]:
list(hola.keys())[-1]

'si'